In [ ]:
!pip install -U transformers accelerate bitsandbytes datasets peft

In [2]:


import numpy as np 
import pandas as pd 


In [ ]:

from huggingface_hub import login

login()

In [5]:
import torch
import json
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

from peft import PeftModel
from huggingface_hub import HfApi, HfFolder

class LLMFinetuner:
    

    def __init__(self, model_name: str, precision: str = "8bit", lora_target_modules: list = None):
        
        self.model_name = model_name
        self.precision = precision
        self.lora_target_modules = lora_target_modules if lora_target_modules is not None else ["q_proj", "v_proj"]

        print(f"Loading model '{self.model_name}' with {self.precision} precision...")
        
        model_dtype = None
        if self.precision == "4bit":
            model_dtype = torch.bfloat16
        elif self.precision == "16bit":
            if torch.cuda.is_available():
                model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            else:
                model_dtype = torch.float32 
            print(f"Using {model_dtype} for 16-bit loading.")


        quantization_config = self._get_quantization_config()

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True,
            torch_dtype=model_dtype
        )

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print("Model and Tokenizer loaded successfully.")


    def _get_quantization_config(self):
        if self.precision == "4bit":
            return BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
        elif self.precision == "8bit":
            return BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,
            )
        else:
            return None
    
    
    def prepare_lora_model(self, r: int = 16, lora_alpha: int = 32, lora_dropout: float = 0.05):

        print("Applying LoRA configuration...")
        lora_config = LoraConfig(
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
            target_modules=self.lora_target_modules
        )

        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()
        print("LoRA model ready.")

    
    def _tokenize_function(self, batch, max_length: int = 512):
        
        full_prompts = [
            f"<s>{p}\n{c}</s>"
            for p, c in zip(batch['prompt'], batch['completion'])
        ]

        tokenized = self.tokenizer(
            full_prompts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors=None
        )
        
        tokenized["labels"] = tokenized["input_ids"].copy()
        
        return tokenized


    def process_dataset(self, raw_data_list: list, max_length: int = 512, batch_size: int = 8):
        
        if not raw_data_list:
            raise ValueError("The raw data list cannot be empty.")
            
        dataset = Dataset.from_list(raw_data_list)
        
        print(f"Tokenizing dataset of size {len(dataset)}...")

        tokenized_dataset = dataset.map(
            lambda batch: self._tokenize_function(batch, max_length),
            batched=True,
            batch_size=batch_size,
            remove_columns=dataset.column_names,
            desc="Tokenizing dataset"
        )
        
        print("Dataset tokenization complete.")
        return tokenized_dataset


    def train(
        self,
        tokenized_train_dataset,
        output_dir: str,
        tokenized_val_dataset=None,
        **kwargs
        ):
        print(f"Starting training, output will be saved to: {output_dir}")
        
        default_training_args = {
            "output_dir": output_dir,
            "per_device_train_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "logging_steps": 20,
            "num_train_epochs": 3,
            "learning_rate": 2e-4,
            "fp16": torch.cuda.is_available(),
            "save_strategy": "epoch",
            "eval_strategy": "epoch" if tokenized_val_dataset is not None else "no",
            "report_to": "none",
            "load_best_model_at_end": True if tokenized_val_dataset is not None else False,
            "metric_for_best_model": "loss",
            "greater_is_better": False,
        }
        
        final_args = {**default_training_args, **kwargs}
        
        training_args = TrainingArguments(**final_args)
        
        data_collator = DataCollatorForLanguageModeling(
            self.tokenizer, 
            mlm=False
        )
        
        trainer = Trainer(
            model=self.model,
            train_dataset=tokenized_train_dataset,
            eval_dataset=tokenized_val_dataset,
            args=training_args,
            data_collator=data_collator,
            tokenizer=self.tokenizer,
        )
        
        trainer.train()
        
        print(f"\nSaving final model and tokenizer to {output_dir}")
        trainer.save_model(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print("Fine-tuning complete.")
    
    


    def save_and_push_adapter(
        self,
        repo_id: str,
        hf_token: str = None,
        push_to_hub: bool = True,
        output_dir: str = "./lora_adapter",
        private: bool = False
    ):
        if not isinstance(self.model, PeftModel):
            raise ValueError("Current model is not a PEFT/LoRA model. Nothing to save.")
    
        print(f"Saving LoRA adapter to: {output_dir}")
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
    
        print("LoRA adapter saved locally.")
    
        if not push_to_hub:
            return
    
        if hf_token is None:
            hf_token = HfFolder.get_token()
    
        if hf_token is None:
            raise ValueError(
                "No Hugging Face token found. Provide hf_token or run 'huggingface-cli login'."
            )
    
        api = HfApi()
    
        print(f"Checking repository: {repo_id}")
    
        try:
            api.create_repo(
                repo_id=repo_id,
                token=hf_token,
                private=private,
                exist_ok=True  
            )
            print(f"Repository ready: {repo_id}")
        except Exception as e:
            raise RuntimeError(f"Failed to create/access repo '{repo_id}'. Error: {str(e)}")
    
        # -----------------------------------------------------
        #  UPLOAD ADAPTER ONLY
        # -----------------------------------------------------
        print(f"Uploading adapter to Hugging Face Hub: {repo_id}")
    
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
            token=hf_token
        )
    
        print("Adapter successfully uploaded to Hugging Face.")

    def merge_and_push_to_hf(
        self,
        repo_id: str,
        output_dir: str = "./merged_model",
        hf_token: str = None,
        private: bool = False
    ):
        from peft import PeftModel
        from huggingface_hub import HfApi, HfFolder
    
        if not isinstance(self.model, PeftModel):
            raise ValueError("Model is not a LoRA PEFT model. Cannot merge.")
    
        print("Merging LoRA weights into the base model...")
        merged_model = self.model.merge_and_unload()
        print(f"Saving merged full model to {output_dir}...")
        merged_model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print("Merged model saved successfully.")
        if hf_token is None:
            hf_token = HfFolder.get_token()
    
        if hf_token is None:
            raise ValueError("HuggingFace token not found. Provide hf_token or run `huggingface-cli login`.")
    
        api = HfApi()
        print(f"Checking or creating repo: {repo_id}")
        api.create_repo(
            repo_id=repo_id,
            token=hf_token,
            private=private,
            exist_ok=True
        )
    
        print(f"Uploading merged model to HuggingFace: {repo_id}")
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            token=hf_token,
            repo_type="model"
        )
    
        print("Merged model successfully uploaded to Hugging Face.")


In [6]:
def build_training_samples(df, title_col="title", content_col="content", label_col="final_labels"):
    samples = []
    
    for _, row in df.iterrows():
        title = str(row.get(title_col, "")).strip()
        content = str(row.get(content_col, "")).strip()
        labels = row.get(label_col, [])

        prompt = f"""Classify the following news: \nNews Title: {title}\nNews Content: {content}
        Give the classified classes in following json format\n
        {{"categories": {{['category1', 'category2']}}}}
        """
        completion = json.dumps({"categories": labels}, ensure_ascii=False)

        samples.append({
            "prompt": prompt,
            "completion": completion
        })
    
    return samples

In [ ]:
if __name__ == "__main__":

    data = pd.read_json("Path to training data").
    val_data = pd.read_json("Path to validation data file").
    data = pd.concat([data, val_data])
    
    example_raw_data = build_training_samples(data[: 3000])
    # print(example_raw_data)
    MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct" 
    # Available precisions: "4bit" (QLoRA), "8bit", "16bit" (BF16/FP16), or "full" (FP32).
    FINETUNE_PRECISION = "8bit" 
    OUTPUT_DIR = "/kaggle/working/Llama-3.2-3B-Instruct"
    
    # Custom LoRA modules if needed, or stick to the default ["q_proj", "v_proj"]
    CUSTOM_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]


    # --- 3. Instantiation and Setup ---
    try:
        finetuner = LLMFinetuner(
            model_name=MODEL_ID,
            precision=FINETUNE_PRECISION,
            lora_target_modules=CUSTOM_TARGET_MODULES
        )

        finetuner.prepare_lora_model(
            r=32,          # Rank
            lora_alpha=64, # Alpha
            lora_dropout=0.1
        )


        # --- 4. Data Processing ---
        training_tokenized_dataset = finetuner.process_dataset(
            raw_data_list=example_raw_data[:2500],
            max_length=512,
            batch_size=16
        )

        validation_tokenized_dataset = finetuner.process_dataset(
            raw_data_list=example_raw_data[2500:],
            max_length=512,
            batch_size=16
        )


        # --- 5. Training ---
        # Note: In a real environment like Kaggle/Colab, ensure output_dir is a full path.
        finetuner.train(
            tokenized_train_dataset=training_tokenized_dataset,
            tokenized_val_dataset=validation_tokenized_dataset,
            output_dir=OUTPUT_DIR,
            # Custom TrainingArguments overrides:
            num_train_epochs=3,
            learning_rate=3e-4,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            logging_steps=5,
        )

        finetuner.merge_and_push_to_hf(
                repo_id="the-usan/llama-3.2-3b-model-testing-v0.1",
                hf_token=#"Your Huggingface token goes here",
                output_dir= "/kaggle/working/merged-full-model".jp
            )


    except Exception as e:
        print(f"\nAn error occurred during the fine-tuning process. This is often due to insufficient GPU memory (OOM). Error: {e}")
        print("Suggestion: Try reducing `per_device_train_batch_size` or switching to '4bit' precision.")
        print("Note: The code above is runnable assuming you have the required libraries (transformers, peft, datasets, accelerate, bitsandbytes) and sufficient GPU memory.")

In [8]:
import json
import torch
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import PeftModel
from datasets import Dataset
from transformers import BitsAndBytesConfig
import unicodedata
import re
import json

def extract_categories_object(text):
    match = re.search(r'\{[^{}]*"categories"[^{}]*\}', text)
    if not match:
        return None  

    json_text = match.group(0)
    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        return None


def clean_label(lbl):
    """
    Strips whitespace and normalizes Unicode characters.
    Works for Urdu or any text.
    """
    if isinstance(lbl, str):
        return unicodedata.normalize("NFC", lbl.strip())
    return lbl

def ensure_list(x):
    """Convert string repr of list to real list if needed."""
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [x] 
    return [x]

class LLMClassifierEvaluator:

    def __init__(
        self,
        base_model_path: str,
        adapter_path: str = None,
        precision: str = "8bit",  
        device: str = None,
        padding_side: str = "left"
    ):
       

        self.precision = precision.lower()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(base_model_path)
        self.tokenizer.padding_side = padding_side
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        quant_config = None
        torch_dtype = None

        if self.precision == "4bit":
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
        elif self.precision == "8bit":
            quant_config = BitsAndBytesConfig(
                load_in_8bit=True
            )
        elif self.precision == "16bit":
            torch_dtype = (
                torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            )
        elif self.precision == "full":
            torch_dtype = torch.float32
        else:
            raise ValueError(f"Invalid precision: {self.precision}")

        if adapter_path is None:
            print(f"Loading base model (precision={self.precision}): {base_model_path}")

            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_path,
                torch_dtype=torch_dtype,
                quantization_config=quant_config,
                device_map="auto"
            )
            self.model.eval()
            return

        print(f"Loading base model for adapter (precision={self.precision}): {base_model_path}")

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            torch_dtype=torch_dtype,
            quantization_config=quant_config,
            device_map="auto"
        )

        print(f"Applying adapter: {adapter_path}")

        try:
            self.model = PeftModel.from_pretrained(
                base_model,
                adapter_path,
                device_map="auto"
            )
        except Exception as e:
            raise RuntimeError(
                f"Failed to load adapter from '{adapter_path}'. "
                f"Make sure it exists locally or on HF. Error:\n{str(e)}"
            )

        self.model.eval()

    def format_prompt(self, title: str, content: str) -> str:
        return (
            f"Classify the following news:\n"
            f"News Title: {title}\n"
            f"News Content: {content}\n"
            # f"Return JSON:\n"
            # f'{{"categories": ["category1", "category2"]}}'
        )

    @torch.inference_mode()
    def predict_batch(self, prompts, max_new_tokens=128):

        encodings = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.model.device)

        outputs = self.model.generate(
            **encodings,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

        decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results = []

        for prompt, text in zip(prompts, decoded):
            raw_output = text
            if text.startswith(prompt):
                text = text[len(prompt):].strip()

            try:
                json_output = json.loads(text)
            except:
                json_output = {"categories": []}

            results.append({
                "raw_output": raw_output,
                "categories": json_output.get("categories", [])
            })

        return results

    def evaluate_dataset(
        self,
        dataset,
        title_col="title",
        content_col="content",
        label_col="final_labels",
        batch_size=8,
        limit=None
    ):
        if limit:
            dataset = dataset.select(range(limit))

        total = len(dataset)
        exact_correct = 0
        iou_scores = []
        iou_50_correct = 0

        results = []

        for start in tqdm(range(0, total, batch_size), desc="Evaluating", ncols=80):
            batch = dataset[start: start + batch_size]
            prompts = [
                self.format_prompt(t, c)
                for t, c in zip(batch[title_col], batch[content_col])
            ]
            true_labels_batch = batch[label_col]

            preds_batch = self.predict_batch(prompts)

            for prompt, true_labels_raw, pred in zip(prompts, true_labels_batch, preds_batch):

                true_labels = ensure_list(true_labels_raw)
                true_set = set(clean_label(x) for x in true_labels)

                raw_output = pred.get("raw_output", "")
                parsed = extract_categories_object(raw_output)
                pred_labels = parsed.get("categories", []) if parsed else []
                pred_set = set(clean_label(x) for x in pred_labels)

                if true_set == pred_set:
                    exact_correct += 1

                if len(true_set | pred_set) > 0:
                    iou = len(true_set & pred_set) / len(true_set | pred_set)
                else:
                    iou = 1.0   # both empty

                iou_scores.append(iou)

                if iou >= 0.5:
                    iou_50_correct += 1

                results.append({
                    "prompt": prompt,
                    "true": list(true_set),
                    "pred": list(pred_set),
                    "raw_output": raw_output,
                    "iou": iou
                })

        # ---------- FINAL METRICS ----------
        exact_acc = exact_correct / total if total else 0
        mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0
        iou_50_acc = iou_50_correct / total if total else 0

        print("\n========== Evaluation Results ==========")
        print(f"Total Samples: {total}")
        print(f"Exact Match Accuracy: {round(exact_acc * 100, 2)}%")
        print(f"Mean IoU Score: {round(mean_iou, 4)}")
        print(f"IoU ≥ 0.5 Accuracy: {round(iou_50_acc * 100, 2)}%")
        print("========================================\n")

        return results

    
    @staticmethod
    def show_errors(results, max_errors=10):
        count = 0
        for r in results:
            if set(r["true"]) != set(r["pred"]):
                print("=" * 60)
                print("True:", r["true"])
                print("Pred:", r["pred"])
                print("Raw Output:", r["raw_output"][:300], "...")
                print("Prompt:", r["prompt"][:200], "...")
                count += 1
                if count >= max_errors:
                    break


In [ ]:
import pandas as pd
df = pd.read_json("Path to test data")
dataset = Dataset.from_pandas(df)

evaluator = LLMClassifierEvaluator(
    base_model_path="the-usan/llama-3.2-3b-model-testing-v0.1",
    precision="8bit"
    
)

results = evaluator.evaluate_dataset(dataset, batch_size=8, limit=1000)
evaluator.show_errors(results)


In [ ]:
evaluator.show_errors(results)[0]

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# =====================================================
# Example input DataFrame
# Replace this with your actual `results` DataFrame
# =====================================================

results = pd.DataFrame(results)

# =====================================================
# SAFETY: ensure every cell is a list
# (silent errors happen here if you skip this)
# =====================================================

def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x) or x is None:
        return []
    return [x]

results["true"] = results["true"].apply(ensure_list)
results["pred"] = results["pred"].apply(ensure_list)

# =====================================================
# MULTI-LABEL BINARIZATION
# =====================================================

mlb = MultiLabelBinarizer()

y_true = mlb.fit_transform(results["true"])
y_pred = mlb.transform(results["pred"])

# =====================================================
# CLASSIFICATION REPORT
# =====================================================

report = classification_report(
    y_true,
    y_pred,
    target_names=mlb.classes_,
    zero_division=0
)

print(report)
